<a href="https://colab.research.google.com/github/naeem182-eng/-Super-AI-Engineer-SS6-Thai-Language-Image-Captioning/blob/main/%5BSuper_AI_Engineer_SS6%5D_Thai_Language_Image_Captioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Download and Unzip

In [ ]:
import os
from google.colab import files

#  Use Kaggle.json
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

#  Dataset

!kaggle datasets download -d isareskhamwirai/data-from-sp-ai-engineer-th-lang-img-captioning
!unzip -qo *.zip -d data/

###1. ติดตั้ง Library ที่จำเป็น

In [ ]:
!pip install transformers torch pillow tqdm deep-translator googletrans==4.0.1 -q

import os
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration
from deep_translator import GoogleTranslator
import warnings
warnings.filterwarnings('ignore')


###2. การตั้งค่าและค้นหาไฟล์แบบ Optimized



In [ ]:
BASE_TEST_DIR = '/content/data/test/test'
SAMPLE_SUB_PATH = '/content/data/sample_submission.csv'

def build_image_index(base_dir):
    """สร้างสารบัญรูปภาพแบบมีประสิทธิภาพ"""
    image_path_map = {}
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_id = os.path.splitext(file)[0]
                try:
                    # แปลงเป็น int แล้วกลับเป็น str เพื่อล้าง leading zeros
                    pure_id = str(int(img_id))
                    image_path_map[pure_id] = os.path.join(root, file)
                except ValueError:
                    image_path_map[img_id] = os.path.join(root, file)
    return image_path_map

print("🔍 กำลังสแกนหาไฟล์รูปภาพ...")
image_path_map = build_image_index(BASE_TEST_DIR)
print(f"✅ พบรูปภาพทั้งหมด {len(image_path_map)} ไฟล์")


###3. โหลด Model พร้อม Optimizations
### ใช้ half precision บน GPU เพื่อลด memory

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 ใช้ Device: {device}")

model_id = "Salesforce/blip-image-captioning-base"

# ใช้ half precision บน GPU เพื่อลด memory
if device.type == "cuda":
    model = BlipForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16
    ).to(device)
    print("✨ ใช้ Half Precision (FP16) บน GPU")
else:
    model = BlipForConditionalGeneration.from_pretrained(model_id).to(device)

processor = BlipProcessor.from_pretrained(model_id)
translator = GoogleTranslator(source='en', target='th')

### 4. Batch Processing และ Caching

In [ ]:
def generate_caption_batch(model, processor, images, device, max_length=30):
    """สร้าง Caption แบบ Batch"""
    inputs = processor(images, return_tensors="pt", padding=True).to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_length,
        num_beams=3,  # Beam search เพื่อ quality ที่ดีขึ้น
        temperature=0.7,  # ความ creative
        do_sample=False  # deterministic
    )
    captions = processor.batch_decode(out, skip_special_tokens=True)
    return captions

def translate_batch(texts, translator, batch_size=50):
    """แปลภาษาแบบ Batch"""
    translated = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        try:
            # กรองค่า None หรือ empty
            batch = [t if t and t.strip() else "no description" for t in batch]
            translated_batch = [translator.translate(t) for t in batch]
            translated.extend(translated_batch)
        except Exception as e:
            print(f"Translation error batch {i}: {e}")
            translated.extend(["ข้อผิดพลาดในการแปล"] * len(batch))
    return translated


### 5. ประมวลผลแบบ Batch พร้อม Error Handling ที่ดีขึ้น

In [ ]:
df_sub = pd.read_csv(SAMPLE_SUB_PATH)
print(f"📸 เริ่มประมวลผล {len(df_sub)} รูปภาพ...")

BATCH_SIZE = 16  # ปรับตาม VRAM ของคุณ
results = []
failed_images = []

model.eval()
with torch.no_grad():
    for i in tqdm(range(0, len(df_sub), BATCH_SIZE), desc="Processing batches"):
        batch_df = df_sub.iloc[i:i+BATCH_SIZE]
        batch_images = []
        batch_ids = []

        # โหลดรูปภาพใน batch
        for _, row in batch_df.iterrows():
            img_id = str(int(row['image_id']))
            img_path = image_path_map.get(img_id)

            if img_path and os.path.exists(img_path):
                try:
                    img = Image.open(img_path).convert('RGB')
                    batch_images.append(img)
                    batch_ids.append(img_id)
                except Exception as e:
                    print(f"Error loading image {img_id}: {e}")
                    batch_images.append(None)
                    batch_ids.append(img_id)
                    failed_images.append(img_id)
            else:
                batch_images.append(None)
                batch_ids.append(img_id)
                failed_images.append(img_id)

        # กรองเฉพาะรูปที่โหลดสำเร็จ
        valid_indices = [idx for idx, img in enumerate(batch_images) if img is not None]
        valid_images = [batch_images[idx] for idx in valid_indices]

        if valid_images:
            # สร้าง Caption ภาษาอังกฤษ
            en_captions = generate_caption_batch(model, processor, valid_images, device)

            # สร้าง mapping สำหรับผลลัพธ์
            batch_results = {}
            for idx, caption in zip(valid_indices, en_captions):
                batch_results[batch_ids[idx]] = caption

            # กรณีที่โหลดไม่สำเร็จ
            for idx, img_id in enumerate(batch_ids):
                if img_id not in batch_results:
                    batch_results[img_id] = "ไม่สามารถโหลดรูปภาพได้"

            # เก็บผลลัพธ์ตามลำดับเดิม
            for img_id in batch_ids:
                results.append(batch_results[img_id])
        else:
            results.extend(["ไม่พบไฟล์ภาพ"] * len(batch_ids))


####6. Batch Translation ---

In [ ]:
def final_clean_translate(texts):
    from deep_translator import GoogleTranslator
    translator = GoogleTranslator(source='en', target='th')
    final_results = []

    print(f"🎯 เริ่มแปลใหม่ยกชุด (Clean Start) อิงสไตล์จาก Val Set...")
    for t in tqdm(texts):
        try:
            # 1. แปลสดจากอังกฤษ
            res = translator.translate(t)

            # 2. ปรับตาม Mapping จาก Val.json และ Train.json
            rules = {
                "สุนัข": "หมา",
                "รับประทาน": "กิน",
                "จักรยานยนต์": "มอเตอร์ไซค์",
                "รถยนต์": "รถ",
                "ขนาดเล็ก": "ตัวเล็ก",
                "รูปภาพของ": "",
                "ภาพของ": "",
            }
            for old, new in rules.items():
                res = res.replace(old, new)

            # 3. คลีนช่องว่าง
            res = " ".join(res.split()).strip()
            final_results.append(res)
        except:
            final_results.append(t)
    return final_results

# รันการแปลใหม่จาก results ภาษาอังกฤษที่มีอยู่
th_captions = final_clean_translate(results)


###7. บันทึกผลลัพธ์

In [ ]:


df_sub['caption'] = th_captions


# บันทึกไฟล์หลายรูปแบบ
df_sub.to_csv('submission_thai_final.csv', index=False)
df_sub.to_csv('submission_thai_backup.csv', index=False)  # backup

# แสดงสถิติ
print("\n" + "="*50)
print("📊 สรุปผลการทำงาน:")
print(f"✅ ประมวลผลสำเร็จ: {len(df_sub) - len(failed_images)} รูป")
print(f"❌ ล้มเหลว: {len(failed_images)} รูป")
if failed_images:
    print(f"  รหัสรูปที่ล้มเหลว: {failed_images[:10]}{'...' if len(failed_images) > 10 else ''}")
print(f"💾 บันทึกไฟล์: submission_thai_final.csv")
print("="*50)

# แสดงตัวอย่างผลลัพธ์
print("\n📝 ตัวอย่างผลลัพธ์ 5 แถวแรก:")
print(df_sub.head())
print("\n✨ เสร็จสมบูรณ์!")

###คำอธิบาย

เลือกใช้ BLIP (Bootstrapping Language-Image Pre-training) ซึ่งเป็น Model ที่โดดเด่นเรื่องการทำความเข้าใจภาพ และมีการจัดการ Workflow ที่ดี เช่นการทำ Batch Processing และการจัดการหน่วยความจำ (Memory Management)

นี่คือการอธิบายการทำงานแยกตามส่วนสำคัญ (Modules)

🛠️ อธิบายการทำงานของ Code
1. การจัดการไฟล์และการทำดัชนี (Image Indexing)
ในส่วนของ build_image_index โค้ดไม่ได้ทำการโหลดรูปทั้งหมดเข้า RAM ทันที (ซึ่งจะทำให้ RAM เต็ม) แต่เลือกที่จะสร้าง Dictionary Map ระหว่าง image_id กับ file_path

Trick: มีการใช้ str(int(img_id)) เพื่อลบเลข 0 ข้างหน้า (Leading Zeros) เช่น 000123 ให้กลายเป็น 123 เพื่อให้ตรงกับ ID ในไฟล์ CSV ของการแข่งขัน

2. การเพิ่มประสิทธิภาพ Model (Optimization)

Half Precision (FP16): การใช้ torch.float16 ช่วยลดการใช้หน่วยความจำบน GPU (VRAM) ลงได้เกือบครึ่งหนึ่ง และทำให้ประมวลผลเร็วขึ้นโดยที่ความแม่นยำแทบไม่ลดลง

Beam Search: ใน generate_caption_batch มีการตั้งค่า num_beams=3 ซึ่งเป็นการให้ Model ลองสุ่มคำตอบหลายๆ ทางแล้วเลือกทางที่ดีที่สุด ทำให้ประโยคภาษาอังกฤษที่ได้มีความไหลลื่นกว่าการสุ่มแบบปกติ (Greedy Search)

3. ขั้นตอนการทำงาน (The Pipeline)
ลำดับการทำงานเป็นแบบ Two-Step Pipeline

Vision-to-Language: ใช้ BLIP อ่านภาพแล้วสร้างคำบรรยายเป็นภาษาอังกฤษ (เพราะ Model ถูกเทรนมาด้วยภาษาอังกฤษเป็นหลัก จะแม่นยำที่สุด)

Translation: ใช้ deep-translator แปลจากอังกฤษเป็นไทย

4. การจัดการ Error และ Batch Processing
Batching: แบ่งการประมวลผลเป็นกลุ่มละ 16 รูป (BATCH_SIZE = 16) เพื่อป้องกัน GPU Error "Out of Memory" (OOM)

Robustness: มีการตรวจสอบ if valid_images: เพื่อให้แน่ใจว่าถ้ามีรูปไหนเสียหรือหาไฟล์ไม่เจอ โปรแกรมจะไม่ค้าง (Crash) แต่จะข้ามไปทำรูปถัดไปแทน